# The Keyboard Tax: GPU run on Google Colab

Runs step 4 (translate) on a free Colab GPU. Step 5 (score) will be added here later.

**Before you start**

1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Have your own Hugging Face token ready (role "Read") and accept the FLORES+ terms once at
   https://huggingface.co/datasets/openlanguagedata/flores_plus
3. Run the cells from top to bottom. If Colab disconnects, run them again from the top:
   finished translations are saved in your Google Drive and are not translated twice.

## 1. Check the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))

## 2. Get the code

After the pull request is merged, change `BRANCH` to `main`.

In [ ]:
BRANCH = "coder2/steps-4-6"

%cd /content
!rm -rf Keyboard_Tax
!git clone -q --branch {BRANCH} https://github.com/karoli-rey/Keyboard_Tax.git
%cd Keyboard_Tax
!git log --oneline -1

## 3. Install the packages

Colab already has PyTorch and transformers. We add only what is missing.

In [ ]:
!pip install -q "spylls==0.1.7" "wordfreq==3.1.1" datasets sacremoses accelerate bitsandbytes
!pip install -q -e . --no-deps

## 4. Log in to Hugging Face

Paste your own token in the box. It is not saved in this notebook.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 5. Keep the results in Google Drive

Everything the pipeline writes goes to the folder `keyboard_tax_outputs` in your Drive,
so a disconnect loses nothing.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/keyboard_tax_outputs
!rm -rf outputs && ln -s /content/drive/MyDrive/keyboard_tax_outputs outputs
!ls outputs

## 6. Steps 1 to 3

Expected at the end: 1012 sentences; `clean` and `stripped_100` 1012 each;
`a_only`, `b_only` and `noise` 422 each.

In [ ]:
!bash scripts/get_dictionary.sh
!python -m keyboard_tax.data
!python -m keyboard_tax.classify
!python -m keyboard_tax.versions

## 7. Step 4: try 20 sentences first

Read a few translations before the full run.

In [ ]:
!python -m keyboard_tax.translate --limit 20 --systems opus_mt nllb

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 120)
tried = pd.read_csv("outputs/4_translations.csv")
tried[tried["sent_id"] == tried["sent_id"].min()]

## 8. Step 4: full run

First the two translation models, then the LLM (the slow one). Each cell can be run again
after a disconnect; it continues where it stopped.

In [ ]:
!python -m keyboard_tax.translate --systems opus_mt nllb

In [ ]:
!python -m keyboard_tax.translate --systems llm

## 9. Check the result

Expected: 3290 rows per system, 0 empty translations, no leftover LLM phrases.

In [ ]:
import pandas as pd

table = pd.read_csv("outputs/4_translations.csv")
print(table.groupby("system").size())
print("empty translations:", int(table["hyp"].fillna("").str.strip().eq("").sum()))
leftover = table["hyp"].fillna("").str.contains(
    r"translation:|here is|here's|^english:", case=False, regex=True
)
print("rows with a possible LLM leftover:", int(leftover.sum()))
table[leftover].head(10)

## 10. Get the files

The results are in your Google Drive, folder `keyboard_tax_outputs`. For step 6 on your
laptop, download that folder and put its files into `Keyboard_Tax/outputs/`.